# 诗歌生成

# 数据处理

In [1]:
import numpy as np
import tensorflow as tf
import collections
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras import layers, optimizers, datasets

start_token = 'bos'
end_token = 'eos'

def process_dataset(fileName):
    examples = []
    with open(fileName, 'r') as fd:
        for line in fd:
            outs = line.strip().split(':')
            content = ''.join(outs[1:])
            ins = [start_token] + list(content) + [end_token] 
            if len(ins) > 200:
                continue
            examples.append(ins)
            
    counter = collections.Counter()
    for e in examples:
        for w in e:
            counter[w]+=1
    
    sorted_counter = sorted(counter.items(), key=lambda x: -x[1])  # 排序
    words, _ = zip(*sorted_counter)
    words = ('PAD', 'UNK') + words[:len(words)]
    word2id = dict(zip(words, range(len(words))))
    id2word = {word2id[k]:k for k in word2id}
    
    indexed_examples = [[word2id[w] for w in poem]
                        for poem in examples]
    seqlen = [len(e) for e in indexed_examples]
    
    instances = list(zip(indexed_examples, seqlen))
    
    return instances, word2id, id2word

def poem_dataset():
    dataset_path = '/home/fanqi/TJ-DeepLearning-Exercise/chap6_RNN/tangshi_for_pytorch/poems.txt'
    
    instances, word2id, id2word = process_dataset(dataset_path)
    ds = tf.data.Dataset.from_generator(lambda: [ins for ins in instances], 
                                        (tf.int64, tf.int64), 
                                        (tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.shuffle(buffer_size=10240)
    ds = ds.padded_batch(100, padded_shapes=(tf.TensorShape([None]),tf.TensorShape([])))
    ds = ds.map(lambda x, seqlen: (x[:, :-1], x[:, 1:], seqlen-1))
    return ds, word2id, id2word

I0000 00:00:1774750841.723611 2646816 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774750841.771039 2646816 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1774750842.683326 2646816 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


# 模型代码， 完成建模代码

In [2]:
class myRNNModel(keras.Model):
    def __init__(self, w2id):
        super(myRNNModel, self).__init__()
        self.v_sz = len(w2id)
        # 将词向量维度和隐藏层神经元翻倍
        self.embed_layer = tf.keras.layers.Embedding(self.v_sz, 128)
        self.rnncell = tf.keras.layers.LSTMCell(256) 
        self.rnn_layer = tf.keras.layers.RNN(self.rnncell, return_sequences=True)
        self.dense = tf.keras.layers.Dense(self.v_sz)
        
    @tf.function
    def call(self, inp_ids):
        x = self.embed_layer(inp_ids)
        x = self.rnn_layer(x)
        logits = self.dense(x)
        return logits
    
    @tf.function
    def get_next_token(self, x, state, temperature=1.0):
        inp_emb = self.embed_layer(x) 
        h, state = self.rnncell(inp_emb, state) 
        logits = self.dense(h) 
        
        # 打破argmax的死板限制
        # Temperature<1: 更加保守求稳，概率最大的字更容易被选中
        # Temperature>1: 更加奔放发散，冷门字也有机会出现
        logits = logits / temperature
        
        out = tf.random.categorical(logits, num_samples=1)
        out = tf.squeeze(out, axis=-1)
        
        return tf.cast(out, tf.int32), state

## 一个计算sequence loss的辅助函数，只需了解用途。

In [3]:
def mkMask(input_tensor, maxLen):
    shape_of_input = tf.shape(input_tensor)
    shape_of_output = tf.concat(axis=0, values=[shape_of_input, [maxLen]])

    oneDtensor = tf.reshape(input_tensor, shape=(-1,))
    flat_mask = tf.sequence_mask(oneDtensor, maxlen=maxLen)
    return tf.reshape(flat_mask, shape_of_output)


def reduce_avg(reduce_target, lengths, dim):
    """
    Args:
        reduce_target : shape(d_0, d_1,..,d_dim, .., d_k)
        lengths : shape(d0, .., d_(dim-1))
        dim : which dimension to average, should be a python number
    """
    shape_of_lengths = lengths.get_shape()
    shape_of_target = reduce_target.get_shape()
    if len(shape_of_lengths) != dim:
        raise ValueError(('Second input tensor should be rank %d, ' +
                         'while it got rank %d') % (dim, len(shape_of_lengths)))
    if len(shape_of_target) < dim+1 :
        raise ValueError(('First input tensor should be at least rank %d, ' +
                         'while it got rank %d') % (dim+1, len(shape_of_target)))

    rank_diff = len(shape_of_target) - len(shape_of_lengths) - 1
    mxlen = tf.shape(reduce_target)[dim]
    mask = mkMask(lengths, mxlen)
    if rank_diff!=0:
        len_shape = tf.concat(axis=0, values=[tf.shape(lengths), [1]*rank_diff])
        mask_shape = tf.concat(axis=0, values=[tf.shape(mask), [1]*rank_diff])
    else:
        len_shape = tf.shape(lengths)
        mask_shape = tf.shape(mask)
    lengths_reshape = tf.reshape(lengths, shape=len_shape)
    mask = tf.reshape(mask, shape=mask_shape)

    mask_target = reduce_target * tf.cast(mask, dtype=reduce_target.dtype)

    red_sum = tf.reduce_sum(mask_target, axis=[dim], keepdims=False)
    red_avg = red_sum / (tf.cast(lengths_reshape, dtype=tf.float32) + 1e-30)
    return red_avg

# 定义loss函数，定义训练函数

In [4]:
@tf.function
def compute_loss(logits, labels, seqlen):
    losses = tf.nn.sparse_softmax_cross_entropy_with_logits(
            logits=logits, labels=labels)
    losses = reduce_avg(losses, seqlen, dim=1)
    return tf.reduce_mean(losses)

@tf.function
def train_one_step(model, optimizer, x, y, seqlen):
    with tf.GradientTape() as tape:
        # 前向传播拿到预测结果
        logits = model(x)
        # 计算损失(这里原代码已经写好了一个带padding mask的compute_loss)
        loss = compute_loss(logits, y, seqlen)

    # 计算梯度并应用更新
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    
    return loss

def train(epoch, model, optimizer, ds):
    loss = 0.0
    accuracy = 0.0
    for step, (x, y, seqlen) in enumerate(ds):
        loss = train_one_step(model, optimizer, x, y, seqlen)

        if step % 500 == 0:
            print('epoch', epoch, ': loss', loss.numpy())

    return loss

# 训练优化过程

In [5]:
optimizer = optimizers.Adam(0.001)
train_ds, word2id, id2word = poem_dataset()
model = myRNNModel(word2id)

model.build(input_shape=(None, None))

for epoch in range(30):
    loss = train(epoch, model, optimizer, train_ds)

I0000 00:00:1774750844.362393 2646816 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46592 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:b3:00.0, compute capability: 8.6
/home/fanqi/my_ai_env/EvaHan2026/lib/python3.12/site-packages/keras/src/layers/layer.py:424: UserWarning: `build()` was called on layer 'my_rnn_model', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
I0000 00:00:1774750845.922640 2646954 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.


epoch 0 : loss 8.820545
epoch 1 : loss 6.258918
epoch 2 : loss 5.867006
epoch 3 : loss 5.5450883
epoch 4 : loss 5.471714
epoch 5 : loss 5.177423
epoch 6 : loss 5.133468
epoch 7 : loss 5.0724635
epoch 8 : loss 4.9600596
epoch 9 : loss 4.9230886
epoch 10 : loss 4.8845916
epoch 11 : loss 4.69622
epoch 12 : loss 4.712133
epoch 13 : loss 4.6131554
epoch 14 : loss 4.597681
epoch 15 : loss 4.6164436
epoch 16 : loss 4.5428267
epoch 17 : loss 4.4655814
epoch 18 : loss 4.5422773
epoch 19 : loss 4.4733486
epoch 20 : loss 4.4052415
epoch 21 : loss 4.4599037
epoch 22 : loss 4.249828
epoch 23 : loss 4.366341
epoch 24 : loss 4.357279
epoch 25 : loss 4.3303275
epoch 26 : loss 4.3154964
epoch 27 : loss 4.3112254
epoch 28 : loss 4.2696495
epoch 29 : loss 4.3107877


# 生成过程

In [6]:
def gen_sentence(temperature=1.0):
    state = [tf.random.normal(shape=(1, 256), stddev=0.5), 
             tf.random.normal(shape=(1, 256), stddev=0.5)]
    
    cur_token = tf.constant([word2id['bos']], dtype=tf.int32)
    collect = []
    
    for _ in range(50):
        cur_token, state = model.get_next_token(cur_token, state, temperature=temperature)
        token_id = cur_token.numpy()[0]
        
        # 如果模型输出了eos(结束符)，就立刻停止作诗，不拖泥带水
        if id2word[token_id] == 'eos':
            break
            
        collect.append(token_id)
        
    return ''.join([id2word[t] for t in collect])

print('【严谨模式(T=0.8)】:', gen_sentence(temperature=0.8))
print('【标准模式(T=1.0)】:', gen_sentence(temperature=1.0))
print('【奔放模式(T=1.2)】:', gen_sentence(temperature=1.2))

【严谨模式(T=0.8)】: 退食无外，门居不愧。松木落花，市上题诗人。年来补壁深，一个缨边尘。
【标准模式(T=1.0)】: 岭荒苍飒险田田，骀荡群书下铁除。买日有情搜尽日，亦应身马只身头。夜闻鼓角徒飞去，沙岛仙人唤不登。和恨
【奔放模式(T=1.2)】: 屐落寒塘宿，西斋肌熟肩。斋折莲杖尾，持浪涧中松。稽面愧黎岳，知非手尚诗。阮趋迟老远，愤不寐家生。杖漱
